In [0]:
import requests

In [0]:
url = "https://huggingface.co/api/models"

In [0]:
parametros = {
    "limit": 10000,
    "full": True,
    "sort": "downloads",
}

In [0]:
resposta = requests.get(url, params=parametros)
print(resposta.status_code)

In [0]:
dados = resposta.json()
print(len(dados))

In [0]:
import json
print(json.dumps(dados[0], indent=2))

In [0]:
#modelos =[]  #cria uma lista vazia pra ser populada pelos 10 mil modelos
#for i in range (10): # começa uma contagem 10 x pra pegar os 10 mil modelos 
  ##  "limit": 10000,
   # "full": True,
   # "sort": "downloads",
   # "offset": i * 10000 # faz o offset pra comecar a partir do 10 mil 
   ## resposta = requests.get(url, params=parametros) # faz a requiscao pra api passando os parametros
  #  dados = resposta.json() # transforma a resposta em json
  #  modelos.extend(dados) # joga os dados na lista 

In [0]:
#lista vazia para armazenar os 10 mil modelos
modelos = [] 
# url inicial e vai mudando a cada volta
url_atual = url 
#cria um array cm os parametros de busca da api
parametros = {   
    "limit": 1000,
    "full": True,
    "sort": "downloads",
}
#vai repetir enquanto tiver a proxima pagina
#e for menor que 10 mil  
#Quando bater 10 mil, para
while url_atual and len(modelos)<10000 :
     resposta = requests.get(url_atual, params=parametros)
     #faz cm oq derrube o codigo na h se o status n for (200)
     resposta.raise_for_status()
     modelos.extend(resposta.json())
     #pega o endereço da próxima página
     url_atual = resposta.links.get("next", {}).get("url")
     #limpa os parametros pra proxima pagina
     parametros = None



In [0]:
print(len(modelos))

In [0]:
ids = set() # define um set vazio pra armazenar os ids
for modeloIA in modelos: # para cada modelo IA na lista modelos
    ids.add( modeloIA["id"] )
print(len(ids))

In [0]:
#bronze pq estamos na parte de receber o dado cru e salvar cru msm
#vai receber uma lista em py e devolver um dataframe
df_bronze = spark.createDataFrame(modelos)

In [0]:
#chama o current pra pegar a data e hora atual
#lit q cria coluna cm pega o valor da variável e coloca igual em todas as 10 mil linhas
# ou seja todas as linhas q vao ter valor igual o lit coloca
from pyspark.sql.functions import current_timestamp, lit

In [0]:
#pega o df_bronze q ja existe e cria uma coluna com o timestamp
df_bronze = df_bronze.withColumn("data_ingestao", current_timestamp()) 

In [0]:
#pega o df_bronze q ja existe 
#coloca e cria uma coluna cm nome origem
# colocando lit pq as linhas q vao ter valor igual
# ai o valor sera oq esta dentro da url (origem)
#pega a url em si ("https://huggingface.co/api/models") q e a origem dos models
df_bronze = df_bronze.withColumn("origem", lit(url))

In [0]:
#select pra pegar so os campos q queremos
#show pra mostrar as 5 linhas 
#trucante pra  n cortar o texto
df_bronze.select("id", "downloads", "data_ingestao", "origem").show(5, truncate=False)

In [0]:
#vai iniciar a escrita salvando no delta
#colocando overwrite caso ja exista , vai substituir
#salvando cm nome bronze_modelos
df_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_modelos")

In [0]:
%sql
SELECT COUNT(*)FROM bronze_modelos

In [0]:
%sql
-- tenta descrever a tabela para encontrar o location e abrir a pasta _delta_log
DESCRIBE DETAIL bronze_modelos

In [0]:
%sql
-- le o _delta_log e devolve em tabela

DESCRIBE HISTORY bronze_modelos

In [0]:
%sql
-- comando para pegar a versao 0 do delta 
-- para olhar todos comparar os arquivos entre as versoes 
SELECT COUNT(*) FROM bronze_modelos VERSION AS OF 0

In [0]:
%sql
--descrever a tabela 
--para ver os schemas e como estao representados os campos dos dados vindos da api 
DESCRIBE TABLE bronze_modelos

In [0]:
%sql
--faz comando sql para pegar dentro das colunas 
-- distinct pega os valores diferentes
-- se tem valores diferentes 
SELECT
  COUNT(DISTINCT modelID) AS modelid,
  COUNT(DISTINCT _id) AS _id,
  COUNT(DISTINCT id) AS id
FROM bronze_modelos

In [0]:
%sql
-- vai olhar oq tem nas colunas
-- limitando a 5 linhas
SELECT _id, id, modelId FROM bronze_modelos LIMIT 5

In [0]:
%sql
-- vai contar quantos tem iguais
--na tabela bronze modelos
--usando as colunas id e modelId
--chamando de total_iguais
-- pra mostrar o total
SELECT COUNT(*) AS total_iguais
FROM bronze_modelos 
WHERE id = modelId;


In [0]:
%sql
--pra ve os nomes das colunas
DESCRIBE TABLE bronze_modelos

In [0]:
%sql
--pega primeiro todas as linhas (count * remove os nulos)
--depois subtrai pela quantidade de linhas que tem nas colubas (junto cm os nulos)
--pra ver quantos nulos tem na coluna 
--pra mostrar o total
SELECT 
 COUNT(*) - COUNT(id) AS nulos_id,
 COUNT(*) - COUNT(author) AS nulos_author,
 COUNT(*) - COUNT(createdAt) AS nulos_createdAt,
 COUNT(*) - COUNT(downloads) AS nulos_downloads,
 COUNT(*) - COUNT(gated) AS nulos_gated,
 COUNT(*) - COUNT(_id) AS nulos__id,
 COUNT(*) - COUNT(lastModified) AS nulos_lastModified,
 COUNT(*) - COUNT(library_name) AS nulos_library_name,
 COUNT(*) - COUNT(likes) AS nulos_likes,
 COUNT(*) - COUNT(modelId) AS nulos_modelId,
 COUNT(*) - COUNT(pipeline_tag) AS nulos_pipeline_tag,
 COUNT(*) - COUNT(private) AS nulos_private,
 COUNT(*) - COUNT(sha) AS nulos_sha,
 COUNT(*) - COUNT(siblings) AS nulos_siblings,
 COUNT(*) - COUNT(tags) AS nulos_tags,
 COUNT(*) - COUNT(data_ingestao) AS nulos_data_ingestao,
 COUNT(*) - COUNT(origem) AS nulos_origem

FROM bronze_modelos

In [0]:
%sql
-- ver oq tem na coluna pipeline_tag da tabela bronze_modelos
SELECT pipeline_tag
FROM bronze_modelos;

In [0]:
%sql
--ver oq tem na coluna gated da tabela bronze_modelos
-- GROUP BY agrupa por uma coluna
-- ver a cardinalidade , se da pra trocar por boolean
SELECT gated, COUNT(*) AS quantidade
FROM bronze_modelos
GROUP BY gated

In [0]:
%sql
-- mostra a tabela inteira 
-- library_name
SELECT library_name
FROM bronze_modelos;

In [0]:
%sql

SELECT library_name ,COUNT(*) as quantidade
FROM bronze_modelos
GROUP BY library_name

In [0]:
%sql
--seleciona a coluna library_name e transforma em minusculo

--para trasnformar as colunas de msm nome em uma so
SELECT lower(library_name) ,COUNT(*) as quantidade
FROM bronze_modelos
GROUP BY lower(library_name)
ORDER BY quantidade DESC 

In [0]:
%sql
--seleciona a coluna library_name
-- usando pra comparar se reduziu a quantidade de valores dps do lower
SELECT library_name ,COUNT(*) as quantidade
FROM bronze_modelos
GROUP BY library_name
ORDER BY quantidade DESC 

In [0]:
%sql
-- seleciona o total de linhas e apelidando de negativo 
-- da tabela bronze modelo
-- so as linhas cm valor negativo da coluna downloads

SELECT COUNT(*) as negativos
FROM bronze_modelos
WHERE downloads <0 
 

In [0]:
%sql
-- seleciona o total de linhas e apelidando de negativo 
-- da tabela bronze modelo
-- so as linhas cm valor negativo da coluna likes
SELECT COUNT(*) as negativos
FROM bronze_modelos
WHERE likes <0 

In [0]:
%sql
-- seleciona todas as linhas e apelidando de futuro
-- da tabela bronze modelo
--ai faz o calculo se tem data no futuro 

SELECT COUNT(*) as futuros
FROM bronze_modelos
WHERE to_timestamp(createdAt) > current_timestamp()


In [0]:
%sql
-- seleciona todas as linhas do created diminuindo das linhas da conversao
-- da tabela bronze modelo
--ai faz o calculo se tem data no futuro 

SELECT COUNT(createdAt)- COUNT(to_timestamp(createdAt)) as nulos
FROM bronze_modelos


In [0]:
%sql
-- seleciona a coluna siblings
-- da tabela bronze modelo
SELECT siblings
FROM bronze_modelos
LIMIT 3

In [0]:
%sql
-- seleciona a coluna modelId e pega o tamanho da lista siblings
-- da tabela bronze modelo
SELECT modelId , size(siblings) AS number_of_files
FROM bronze_modelos
LIMIT 3

In [0]:
%sql 

SELECT _id as id
FROM bronze_modelos
LIMIT 5

In [0]:
%sql
SELECT modelID as model_id
FROM bronze_modelos
LIMIT 5

In [0]:
%sql
SELECT createdAt as created_at
FROM bronze_modelos
LIMIT 5

In [0]:
%sql
SELECT lastModified as last_modified
FROM bronze_modelos
LIMIT 5

In [0]:
%sql
-- seleciona a coluna modelid da bronze modelos
-- olhando se contem algum valor com pt ou text_generation
--para ve se explode e repete o msm modelo
SELECT DISTINCT  modelId
FROM bronze_modelos 
WHERE array_contains(tags, 'pt') OR array_contains(tags , 'text_generation')
 

In [0]:
%sql
SELECT tags
FROM bronze_modelos
WHERE array_contains(tags, 'pt') OR array_contains(tags , 'text_generation')
    
-- seleciona a coluna tags da bronze modelos
-- olhando se contem algum valor com pt ou text_generation
--para ve se retorna como array ou explodido q sao varias linhas com o msm modelo e diferentes tags

In [0]:
%sql
-- para ver todas as colunas e descidir quais mudar na transformação da bronze
SELECT * FROM bronze_modelos